### Data Preparation

In [1]:
import torch
import time
import numpy as np
import pandas as pd
import psutil
import gc
import os

from abc import ABC, abstractmethod
from dataclasses import dataclass
from datasets import load_dataset
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from transformers import pipeline

In [2]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

dataset = load_dataset(
    "papluca/language-identification"
)

### Evaluation Setup

In [3]:
class LanguageDetector(ABC):
    @abstractmethod
    def predict(self, texts: list[str]) -> list[str]:
        pass

    @abstractmethod
    def train(self, texts, labels):
        pass

In [4]:
@dataclass
class EvaluationResult:
    model_name: str
    accuracy: float
    macro_f1: float
    avg_inference_ms: float
    per_language_accuracy: dict[str, float]

In [5]:
def evaluate(model, model_name: str, texts: list[str], labels: list[str]) -> EvaluationResult:
    start_time = time.perf_counter()
    predictions = model.predict(texts)
    total_time = time.perf_counter() - start_time

    avg_inference_ms = (total_time / len(texts)) * 1000

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    per_language_accuracy = {}
    for language in sorted(set(labels)):
        indices = [i for i, label in enumerate(labels) if label == language]
        y_true = [labels[i] for i in indices]
        y_pred = [predictions[i] for i in indices]
        per_language_accuracy[language] = accuracy_score(y_true, y_pred)

    return EvaluationResult(
        model_name=model_name,
        accuracy=accuracy,
        macro_f1=macro_f1,
        avg_inference_ms=avg_inference_ms,
        per_language_accuracy=per_language_accuracy
    )

In [6]:
class Leaderboard:
    def __init__(self):
        self.results = []

    def add(self, result: EvaluationResult):
        self.results.append(result)

    def dataframe(self):
        rows = []
        for result in self.results:
            rows.append({
                "Model": result.model_name,
                "Accuracy": round(result.accuracy, 3),
                "Macro F1": round(result.macro_f1, 3),
                "Avg Inference Time (ms)": round(result.avg_inference_ms, 3)
            })
        df = pd.DataFrame(rows)
        return df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
leaderboard = Leaderboard()

In [7]:
def clean_memory():
    """Free as much RAM and VRAM as possible."""
    gc.collect()
    print("Memory cleanup completed.")
    print(f"RAM Usage : {psutil.virtual_memory().percent:.1f}%")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        total_vram = torch.cuda.get_device_properties(0).total_memory
        reserved_vram = torch.cuda.memory_reserved()
        print(f"VRAM Usage: {(100 * reserved_vram / total_vram):.1f}%")

## Models

### 1. Simple Logistic Regression

In [8]:
class CharNGramLogisticDetector(LanguageDetector):
    def __init__(self):
        self.pipeline = Pipeline([
            ("vectorizer", TfidfVectorizer(analyzer="char", ngram_range=(2, 4))),
            ("classifier", LogisticRegression(max_iter=1000))
        ])

    def train(self, texts, labels):
        self.pipeline.fit(texts, labels)

    def predict(self, texts) -> list[str]:
        return self.pipeline.predict(texts)

In [ ]:
log_reg_model = CharNGramLogisticDetector()
log_reg_model.train(dataset["train"]["text"], dataset["train"]["labels"])
print(f"TF/IDF prodcuced {len(log_reg_model.pipeline.named_steps["vectorizer"].vocabulary_)} features")

TF/IDF prodcuced 1160712 features


In [ ]:
leaderboard.add(
    evaluate(
        model=log_reg_model,
        model_name="Char N-Gram + Logistic Regression",
        texts=dataset["test"]["text"],
        labels=dataset["test"]["labels"]
    )
)
leaderboard.dataframe()

,Model,Accuracy,Macro F1,Avg Inference Time (ms)
0,Char N-Gram + Logistic Regression,0.992,0.992,0.261


In [ ]:
clean_memory()

Memory cleanup completed.
RAM Usage : 60.0%
VRAM Usage: 0.0%


### 2. Character N-Grams + Linear SVM

In [22]:
class CharNGramSVMDetector(LanguageDetector):
    def __init__(self):
        self.pipeline = Pipeline([
            ("vectorizer", TfidfVectorizer(analyzer="char", ngram_range=(2, 4))),
            ("classifier", LinearSVC())
        ])

    def train(self, texts, labels):
        self.pipeline.fit(texts, labels)

    def predict(self, texts) -> list[str]:
        return self.pipeline.predict(texts)

In [23]:
svm_model = CharNGramSVMDetector()
svm_model.train(dataset["train"]["text"], dataset["train"]["labels"])
print(f"TF/IDF prodcuced {len(svm_model.pipeline.named_steps["vectorizer"].vocabulary_)} features")

TF/IDF prodcuced 1160712 features


In [24]:
leaderboard.add(
    evaluate(
        model=svm_model,
        model_name="Char N-Gram + Linear SVM",
        texts=dataset["test"]["text"],
        labels=dataset["test"]["labels"]
    )
)
leaderboard.dataframe()

,Model,Accuracy,Macro F1,Avg Inference Time (ms)
0,XLM-R Language Detection,0.996,0.996,8.328
1,XLM-R Language Detection,0.996,0.996,8.306
2,Char N-Gram + Linear SVM,0.995,0.995,0.196
3,Char N-Gram + Linear SVM,0.995,0.995,0.206
4,Char N-Gram + Logistic Regression,0.992,0.992,0.261


In [ ]:
clean_memory()

Memory cleanup completed.
RAM Usage : 65.7%
VRAM Usage: 0.0%


### 3. SoTA Model: xlm-roberta-base-language-detection

In [18]:
class XLMRobertaLanguageDetector(LanguageDetector):
    def __init__(self):
        self.pipe = pipeline(
            "text-classification",
            model="papluca/xlm-roberta-base-language-detection",
            device=0
        )

    def train(self, texts, labels):
        pass

    def predict(self, texts: list[str]) -> list[str]:
        results = self.pipe(
            list(texts),
            top_k=1,
            truncation=True)
        return [result[0]["label"] for result in results]

In [20]:
leaderboard.add(
    evaluate(
        model=XLMRobertaLanguageDetector(),
        model_name="XLM-R Language Detection",
        texts=list(dataset["test"]["text"]),
        labels=dataset["test"]["labels"]
    )
)
leaderboard.dataframe()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,Model,Accuracy,Macro F1,Avg Inference Time (ms)
0,XLM-R Language Detection,0.996,0.996,8.306
1,XLM-R Language Detection,0.996,0.996,8.328
2,Char N-Gram + Linear SVM,0.995,0.995,0.196
3,Char N-Gram + Logistic Regression,0.992,0.992,0.261


In [21]:
clean_memory()

Memory cleanup completed.
RAM Usage : 79.5%
VRAM Usage: 0.5%


## Save Best Model

In [ ]:
import joblib

joblib.dump(svm_model.pipeline, "../models/language_detector_svm.joblib")

['models/language_detector_svm.joblib']